In [1]:
import os, sys

def find_project_root(marker="config/datasets.yaml"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.common.spark_session import get_spark_session
from src.ml.config import load_ml_config
from src.ml.training_dataset import generate_training_dataset, load_training_dataset

spark = get_spark_session("ml-training-dataset")
cfg = load_ml_config()

26/10/01 15:25:36 WARN Utils: Your hostname, Thilinis-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.8 instead (on interface en0)
26/10/01 15:25:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 15:25:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import time, json
t0 = time.time()
manifest = generate_training_dataset(spark, cfg)
print(f"Generated in {time.time() - t0:.1f}s")
print(json.dumps({k: manifest[k] for k in ("time_range_utc", "split_boundaries_utc", "splits", "source_gold_version")}, indent=2))

Generated in 7.8s
{
  "time_range_utc": {
    "start": "2024-01-01 05:00:00",
    "end": "2024-04-01 03:00:00"
  },
  "split_boundaries_utc": {
    "train_end": "2024-03-04 21:00:00",
    "validation_end": "2024-03-18 12:00:00"
  },
  "splits": {
    "train": {
      "rows": 301920,
      "zones": 222,
      "mean_target": 18.9753,
      "first_hour": "2024-01-08 05:00:00",
      "last_hour": "2024-03-04 20:00:00"
    },
    "validation": {
      "rows": 72594,
      "zones": 222,
      "mean_target": 22.1587,
      "first_hour": "2024-03-04 21:00:00",
      "last_hour": "2024-03-18 11:00:00"
    },
    "test": {
      "rows": 72816,
      "zones": 222,
      "mean_target": 21.5969,
      "first_hour": "2024-03-18 12:00:00",
      "last_hour": "2024-04-01 03:00:00"
    }
  },
  "source_gold_version": 1
}


In [4]:
from pyspark.sql import functions as F
data = load_training_dataset(spark, cfg)
data.printSchema()
data.orderBy("pickup_location_id", "hour_ts").limit(10).toPandas()

root
 |-- pickup_location_id: integer (nullable = true)
 |-- hour_ts: timestamp_ntz (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- weather_temperature: double (nullable = true)
 |-- weather_relative_humidity: double (nullable = true)
 |-- weather_precipitation: double (nullable = true)
 |-- weather_wind_speed: double (nullable = true)
 |-- weather_pressure: double (nullable = true)
 |-- weather_condition_code: integer (nullable = true)
 |-- air_quality_pm25: double (nullable = true)
 |-- air_quality_station_count: long (nullable = true)
 |-- demand_lag_1h: integer (nullable = true)
 |-- demand_lag_24h: integer (nullable = true)
 |-- demand_lag_168h: integer (nullable = true)
 |-- demand_rolling_mean_24h: double (nullable = true)
 |-- trip_count: integer (nullable = true)
 |-- split: string (nullable = true)



,pickup_location_id,hour_ts,pickup_zone,pickup_borough,weather_temperature,weather_relative_humidity,weather_precipitation,weather_wind_speed,weather_pressure,weather_condition_code,air_quality_pm25,air_quality_station_count,demand_lag_1h,demand_lag_24h,demand_lag_168h,demand_rolling_mean_24h,trip_count,split
0,1,2024-01-08 05:00:00,Newark Airport,EWR,2.8,64.0,0.0,27.7,1015.7,2,NaN,NaN,0,0,0,0.625000,0,train
1,1,2024-01-08 06:00:00,Newark Airport,EWR,2.8,64.0,0.0,33.5,1016.5,3,NaN,NaN,0,0,0,0.625000,0,train
2,1,2024-01-08 07:00:00,Newark Airport,EWR,2.8,61.0,0.0,20.5,1017.9,3,NaN,NaN,0,0,0,0.625000,0,train
3,1,2024-01-08 08:00:00,Newark Airport,EWR,3.3,62.0,0.0,18.4,1019.7,2,NaN,NaN,0,0,0,0.625000,0,train
4,1,2024-01-08 09:00:00,Newark Airport,EWR,3.3,59.0,0.0,22.3,1020.2,2,NaN,NaN,0,1,0,0.625000,0,train
5,1,2024-01-08 10:00:00,Newark Airport,EWR,3.3,54.0,0.0,27.7,1021.1,4,NaN,NaN,0,0,1,0.583333,2,train
6,1,2024-01-08 11:00:00,Newark Airport,EWR,2.8,59.0,0.0,25.9,1022.7,2,NaN,NaN,2,0,4,0.666667,0,train
7,1,2024-01-08 12:00:00,Newark Airport,EWR,3.3,54.0,0.0,25.9,1023.8,2,NaN,NaN,0,1,0,0.666667,0,train
8,1,2024-01-08 13:00:00,Newark Airport,EWR,3.3,57.0,0.0,27.7,1024.6,2,NaN,NaN,0,0,0,0.625000,0,train
9,1,2024-01-08 14:00:00,Newark Airport,EWR,3.3,57.0,0.0,27.7,1026.3,2,NaN,NaN,0,1,0,0.625000,2,train


In [5]:
# target distribution and share of zero-demand hours per split
data.groupBy("split").agg(
    F.count("*").alias("rows"),
    F.round(F.avg(cfg.target), 2).alias("mean_trips"),
    F.round(F.stddev(cfg.target), 2).alias("std_trips"),
    F.round(F.avg((F.col(cfg.target) == 0).cast("int")), 3).alias("share_zero_hours"),
).toPandas()

,split,rows,mean_trips,std_trips,share_zero_hours
0,train,301920,18.98,54.48,0.530
1,validation,72594,22.16,61.62,0.475
2,test,72816,21.60,59.18,0.464


In [6]:
# missing values left for the Task 2 imputation step
data.select([F.round(F.avg(F.col(c).isNull().cast("int")), 4).alias(c) for c in data.columns]).toPandas().T.rename(columns={0: "share_null"})

26/10/01 15:26:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,share_null
pickup_location_id,0.0000
hour_ts,0.0000
pickup_zone,0.0000
pickup_borough,0.0000
weather_temperature,0.0000
weather_relative_humidity,0.0000
weather_precipitation,0.0774
weather_wind_speed,0.0000
weather_pressure,0.0000
weather_condition_code,0.0000


In [ ]:
spark.stop()